1. Um nível de indentação por método

In [1]:
# ❌ Ruim
def imprimir_pedidos(pedidos):
    for pedido in pedidos:
        if pedido.ativo:
            print(pedido.nome)

# ✅ Bom
def imprimir_pedidos(pedidos):
    for pedido in pedidos:
        imprimir_se_ativo(pedido)

def imprimir_se_ativo(pedido):
    if pedido.ativo:
        print(pedido.nome)

2. Não use else

In [ ]:
# ❌ Ruim
def classificar(idade):
    if idade < 18:
        return "menor"
    else:
        return "adulto"

# ✅ Bom
def classificar(idade):
    if idade < 18:
        return "menor"
    return "adulto"

3. Envolva primitivos em objetos

In [ ]:
# ❌ Ruim
def processar(cep: str, valor: float):
    ...

# ✅ Bom
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True)
class Cep:
    valor: str

    def __post_init__(self):
        if len(self.valor) != 8 or not self.valor.isdigit():
            raise ValueError(f"CEP inválido: {self.valor}")

@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Valor não pode ser negativo")

    def somar(self, outro: "Dinheiro") -> "Dinheiro":
        return Dinheiro(self.valor + outro.valor)

def processar(cep: Cep, valor: Dinheiro):
    ...

4. Coleções de primeira classe

In [ ]:
# ❌ Ruim
class Pedido:
    def __init__(self):
        self.itens = []
        self.status = "aberto"

# ✅ Bom
class Itens:
    def __init__(self, itens: list):
        self._itens = itens

    def total(self) -> Dinheiro:
        return sum((i.preco for i in self._itens), Dinheiro(Decimal(0)))

    def ativos(self) -> "Itens":
        return Itens([i for i in self._itens if i.ativo])

    def __iter__(self):
        return iter(self._itens)

5. Um ponto por linha (Lei de Demeter)

In [ ]:
# ❌ Ruim
cidade = pedido.cliente.endereco.cidade

# ✅ Bom
class Pedido:
    def cidade_do_cliente(self) -> str:
        return self.cliente.cidade()

class Cliente:
    def cidade(self) -> str:
        return self.endereco.cidade()

6. Não abrevie nomes

In [ ]:
# ❌ Ruim
class OrdProc:
    def proc_itns(self, itns):
        ...

# ✅ Bom
class ProcessadorDePedido:
    def processar_itens(self, itens: Itens):
        ...

7. Entidades pequenas
Em Python isso se traduz em manter classes com até ~50 linhas e módulos coesos. Se uma classe está crescendo, é um sinal para decompor:

In [3]:
# ❌ Ruim — uma classe fazendo tudo
class Pedido:
    def calcular_total(self): ...
    def aplicar_desconto(self): ...
    def enviar_email(self): ...
    def salvar_no_banco(self): ...
    def gerar_nota_fiscal(self): ...

# ✅ Bom — responsabilidades separadas
class Pedido: ...
class CalculadoraDeDesconto: ...
class NotificacaoDePedido: ...
class RepositorioDePedido: ...
class EmissorDeNotaFiscal: ...

8. No máximo dois atributos de instância

In [ ]:
# ❌ Ruim
class Cliente:
    def __init__(self, nome, sobrenome, email, telefone):
        self.nome = nome
        self.sobrenome = sobrenome
        self.email = email
        self.telefone = telefone

# ✅ Bom
@dataclass(frozen=True)
class NomeCompleto:
    primeiro: str
    sobrenome: str

    def completo(self) -> str:
        return f"{self.primeiro} {self.sobrenome}"

@dataclass(frozen=True)
class Contato:
    email: str
    telefone: str

@dataclass
class Cliente:
    nome: NomeCompleto
    contato: Contato

9. Sem getters/setters públicos — exponha comportamento

In [ ]:
# ❌ Ruim
if pedido.get_status() == "pago":
    pedido.set_status("enviado")

# ✅ Bom
class Pedido:
    def __init__(self):
        self._status = "aberto"

    def pagar(self):
        if self._status != "aberto":
            raise ValueError("Pedido não pode ser pago neste estado")
        self._status = "pago"

    def enviar(self):
        if self._status != "pago":
            raise ValueError("Pedido precisa estar pago para ser enviado")
        self._status = "enviado"

    def esta_enviado(self) -> bool:
        return self._status == "enviado"

# Uso
pedido.pagar()
pedido.enviar()

In [4]:
from dataclasses import dataclass
from decimal import Decimal


@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Valor não pode ser negativo")

    def somar(self, outro: "Dinheiro") -> "Dinheiro":
        return Dinheiro(self.valor + outro.valor)


@dataclass(frozen=True)
class Item:
    nome: str
    preco: Dinheiro


class Itens:
    def __init__(self, itens: list[Item]):
        self._itens = itens

    def total(self) -> Dinheiro:
        return sum(
            (i.preco for i in self._itens),
            Dinheiro(Decimal(0))
        )

    def __iter__(self):
        return iter(self._itens)


class Pedido:
    def __init__(self, itens: Itens):
        self._itens = itens
        self._status = "aberto"

    def pagar(self):
        if self._status != "aberto":
            raise ValueError("Pedido já foi processado")
        self._status = "pago"

    def total(self) -> Dinheiro:
        return self._itens.total()

    def esta_pago(self) -> bool:
        return self._status == "pago"